# Test `RSMDataloader_CMS_Tiled`

This notebook exercises the Tiled-based CMS loader in `napari_resview.data_io`.

Because `tiled` may not be installed and no live server is available offline, the
loader is tested against a small **in-memory mock catalog** that mimics the parts
of a Tiled / Bluesky run that the loader touches:

- `catalog[key]` / iteration over keys
- `run.metadata['start']['scan_id']`
- `run['primary']['data'][<detector>_image]` frames
- `run['primary']['data'][<motor>]` angle arrays

The loader accepts a live `catalog` node directly, which makes this mock a
faithful stand-in. A real-server example (`from_uri`) is included at the end.

## 1. Import the loader

We register a lightweight `napari_resview` package stub before importing so that
the heavy Qt/napari widget in `__init__.py` is **not** imported — only `data_io`
and its `spec_parser` dependency are loaded.

In [ ]:
import pathlib
import sys
import types

# Locate the repo 'src' directory regardless of where this notebook lives.
_here = pathlib.Path.cwd()
_repo = _here
while _repo != _repo.parent and not (_repo / "src" / "napari_resview").exists():
    _repo = _repo.parent
_src = _repo / "src"
assert (_src / "napari_resview").exists(), f"Could not locate src/ from {_here}"
sys.path.insert(0, str(_src))

# Pre-register a bare 'napari_resview' package so importing the submodule does
# NOT execute the real __init__.py (which pulls in Qt widgets).
if "napari_resview" not in sys.modules:
    _pkg = types.ModuleType("napari_resview")
    _pkg.__path__ = [str(_src / "napari_resview")]
    sys.modules["napari_resview"] = _pkg

from napari_resview.data_io import RSMDataloader_CMS_Tiled, TILED_AVAILABLE

print("loader imported OK")
print("tiled installed:", TILED_AVAILABLE)

## 2. Minimal `ExperimentSetup` YAML

The loader reads detector geometry from a YAML file. We write a small temporary
one matching the mock detector size (16×16).

In [ ]:
import tempfile

SETUP_YAML = """
ExperimentSetup:
  distance: 3.03
  pitch: 0.000172
  ycenter: 8
  xcenter: 8
  xpixels: 16
  ypixels: 16
  energy: 13.49
"""

_tmp = tempfile.NamedTemporaryFile(
    "w", suffix=".yaml", delete=False, encoding="utf-8"
)
_tmp.write(SETUP_YAML)
_tmp.close()
setup_yaml_path = _tmp.name
print("wrote setup YAML ->", setup_yaml_path)

## 3. Build a mock Tiled catalog

These tiny classes replicate only the access patterns the loader uses.

In [ ]:
import numpy as np


class MockArray:
    """Wraps an ndarray, exposing `.shape` and `[:]` slicing."""

    def __init__(self, arr):
        self._arr = np.asarray(arr)

    @property
    def shape(self):
        return self._arr.shape

    def __getitem__(self, item):
        return self._arr[item]


class MockData:
    """Dict-like 'data' container (supports `in`, iteration, `[key]`)."""

    def __init__(self, mapping):
        self._m = {k: MockArray(v) for k, v in mapping.items()}

    def __contains__(self, key):
        return key in self._m

    def __iter__(self):
        return iter(self._m)

    def __getitem__(self, key):
        return self._m[key]


class MockStream:
    def __init__(self, data):
        self._data = data

    def __contains__(self, key):
        return key == "data"

    def __getitem__(self, key):
        if key == "data":
            return self._data
        raise KeyError(key)


class MockRun:
    def __init__(self, streams, metadata):
        self._streams = streams
        self.metadata = metadata

    def __contains__(self, key):
        return key in self._streams

    def __getitem__(self, key):
        return self._streams[key]


class MockCatalog:
    def __init__(self, runs):
        self._runs = dict(runs)

    def __iter__(self):
        return iter(self._runs)

    def __getitem__(self, key):
        return self._runs[key]


def make_run(scan_id, n_frames=5, shape=(16, 16), th_step=0.5, tth=20.0,
             detector="pilatus2M_image"):
    rng = np.random.default_rng(scan_id)
    imgs = rng.integers(0, 100, size=(n_frames,) + shape).astype(np.uint32)
    th = th_step * np.arange(n_frames, dtype=float)
    data = MockData({
        detector: imgs,
        "th": th,
        "tth": np.full(n_frames, tth, dtype=float),
        "chi": np.zeros(n_frames, dtype=float),
        "phi": np.zeros(n_frames, dtype=float),
    })
    metadata = {"start": {"scan_id": int(scan_id), "plan_name": "rock"}}
    return MockRun({"primary": MockStream(data)}, metadata)


catalog = MockCatalog({
    "uid-aaa": make_run(1001, n_frames=5, th_step=0.5, tth=20.0),
    "uid-bbb": make_run(1002, n_frames=4, th_step=0.5, tth=20.0),
    "uid-ccc": make_run(1003, n_frames=6, th_step=0.5, tth=20.0),
})
print("mock catalog keys:", list(catalog))

## 4. Load every run

Pass the mock `catalog` directly. The loader auto-detects the `*_image` detector
field and reads the `th/tth/chi/phi` motors.

In [ ]:
loader = RSMDataloader_CMS_Tiled(setup_yaml_path, catalog=catalog)
setup, ub, df = loader.load()

print("setup:", setup)
print("UB:\n", ub)
print("\nframes loaded:", len(df))
df.drop(columns=["intensity"]).assign(
    img_shape=[a.shape for a in df["intensity"]]
)

In [ ]:
# Basic assertions on the merged result.
assert list(df.columns) == [
    "scan_number", "intensity", "tth", "th", "chi", "phi"
]
assert len(df) == 5 + 4 + 6, "expected total frames across all runs"
assert set(df["scan_number"]) == {1001, 1002, 1003}
assert np.allclose(df["tth"], 20.0)
assert np.allclose(df["chi"], 0.0) and np.allclose(df["phi"], 0.0)
assert df["intensity"].iloc[0].shape == (16, 16)
assert ub.shape == (3, 3)
print("All assertions passed ✔")

## 5. `selected_scans` filtering

Selecting by integer `scan_id` matches each run's `start.scan_id` metadata.

In [ ]:
loader_sel = RSMDataloader_CMS_Tiled(
    setup_yaml_path, catalog=catalog, selected_scans=[1001, 1003]
)
_, _, df_sel = loader_sel.load()

print("selected scan numbers:", sorted(set(df_sel["scan_number"])))
assert set(df_sel["scan_number"]) == {1001, 1003}
assert len(df_sel) == 5 + 6
print("selected_scans OK ✔  (", len(df_sel), "frames )")

## 6. `crop_window` and `motor_map`

- `crop_window=((r0, r1), (c0, c1))` crops every frame.
- `motor_map` overrides which Tiled field maps to each canonical angle.

In [ ]:
loader_crop = RSMDataloader_CMS_Tiled(
    setup_yaml_path,
    catalog=catalog,
    selected_scans=1002,
    crop_window=((2, 10), (3, 11)),
    motor_map={"th": "th", "tth": "tth"},
)
_, _, df_crop = loader_crop.load()

shapes = {a.shape for a in df_crop["intensity"]}
print("cropped frame shapes:", shapes)
assert shapes == {(8, 8)}, "crop_window should yield 8x8 frames"
assert set(df_crop["scan_number"]) == {1002}
print("crop_window + motor_map OK ✔")
df_crop.drop(columns=["intensity"])

## 7. Visualize a loaded frame

In [ ]:
import matplotlib.pyplot as plt

frame = df["intensity"].iloc[0]
plt.figure(figsize=(4, 4))
plt.imshow(frame, cmap="inferno")
plt.title(f"scan {int(df['scan_number'].iloc[0])} · th={df['th'].iloc[0]:.2f}")
plt.colorbar(shrink=0.8)
plt.tight_layout()
plt.show()

## 8. Using a real Tiled server

Against a live NSLS-II Tiled deployment the loader connects on its own — just
give it a `uri` (and, if needed, an `api_key` and `catalog_path`). Requires
`pip install "tiled[client]"`.

```python
loader = RSMDataloader_CMS_Tiled(
    setup_yaml_path,
    uri="https://tiled.nsls2.bnl.gov",
    api_key="<YOUR_API_KEY>",   # or omit for public/anon access
    catalog_path=("cms", "raw"),
    selected_scans=[796715, 796716],
    detector="pilatus2M_image",  # optional; auto-detected otherwise
)
setup, ub, df = loader.load()
```

The cell below runs that example only if `tiled` is installed and you fill in a
URI.

In [ ]:
REAL_URI = ""  # e.g. "https://tiled.nsls2.bnl.gov"

if TILED_AVAILABLE and REAL_URI:
    loader = RSMDataloader_CMS_Tiled(
        setup_yaml_path,
        uri=REAL_URI,
        catalog_path=("cms", "raw"),
        selected_scans=None,  # set to specific scan_ids for a real run
    )
    setup, ub, df_real = loader.load()
    print("loaded", len(df_real), "frames from", REAL_URI)
else:
    print("Skipped: set REAL_URI and install tiled[client] to run this cell.")